Intento de detección de matrículas basada en contornos.

In [ ]:
import cv2  
import math 

from ultralytics import YOLO

# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores

#Para un vídeo
filename = "C0142.mp4"

cap = cv2.VideoCapture(filename)

cv2.namedWindow('Deteccion con YOLO', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Deteccion con YOLO', 1280, 720)

# funcion para detectar matrículas
def detect_plate(car_region):
    # paso la imagen a escala de grises
    gris = cv2.cvtColor(car_region, cv2.COLOR_BGR2GRAY)

    # suavizo para reducir el ruido
    gris = cv2.GaussianBlur(gris, (5, 5), 0)

    # aplico umbral adaptativo para destacar los bordes
    img_th1 = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 5) 

    # obtengo los contornos externos
    contornos, _ = cv2.findContours(img_th1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # comprobamos los contornos
    candidatos_mat = []
    for contorno in contornos:
        # aproximo el contorno a un polígono
        perimeter = cv2.arcLength(contorno, True)
        aprox = cv2.approxPolyDP(contorno, 0.018 * perimeter, True)

        # obtengo el rectángulo
        x, y, w, h = cv2.boundingRect(aprox)

        # características para identificar la matrícula
        aspect_ratio = w / float(h)
        area = w * h
        car_area = car_region.shape[0] * car_region.shape[1]
        relative_area = area / car_area

        if (2.0 <= aspect_ratio <= 5.5 and 
            0.01 <= relative_area <= 0.15 and
            w > 40 and h > 10):

            # Puntuación basada en qué tan cerca está del ratio ideal
            ideal_ratio = 4.5
            ratio_score = 1 - abs(aspect_ratio - ideal_ratio) / ideal_ratio
            
            candidatos_mat.append({
                'contour': aprox,
                'bbox': (x, y, w, h),
                'score': ratio_score * relative_area,
                'aspect_ratio': aspect_ratio
            })

    # cogemos el mejor candidato
    if candidatos_mat:
        mejor_matricula = max(candidatos_mat, key=lambda x: x['score'])
        return mejor_matricula

while cap.isOpened():
    ret, frame = cap.read()

    # si no hay imagen salimos
    if not ret:
        break

    # se ejecuta el modelo en el frame y se añaden los recuadros
    results = model(frame, classes=[2, 3, 5, 7], conf=0.5)
    annotated_frame = results[0].plot()

    # busco las matrículas de cada vehículo
    for result in results[0].boxes.data:
        x1, y1, x2, y2, conf, cls = result
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # Extraer región del vehículo
        car_region = frame[y1:y2, x1:x2].copy()

        if car_region.size > 0:
            # busco la matricula
            plate = detect_plate(car_region)
            
            # si la encuentro, la dibujo
            if plate is not None:
                detections_count += 1
                px, py, pw, ph = plate['bbox']
                
                # Ajustar coordenadas al frame completo
                px += x1
                py += y1
                
                # Dibujar rectángulo de la matrícula en rojo
                cv2.rectangle(annotated_frame, (px, py), (px + pw, py + ph), (0, 0, 255), 2)
                cv2.putText(annotated_frame, 'PLATE', (px, py - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    frame = annotated_frame

    cv2.imshow('Deteccion con YOLO', annotated_frame)

    # se sale con ESC o Q/q
    key = cv2.waitKey(1)
    print(key)
    if key == 27 or key == 81 or key == 113:
        break

cap.release()
cv2.destroyAllWindows()

Modelo YOLO para matrículas

In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado
model = YOLO("yolo11n.pt")

# Entrenar
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo_matriculas",
    device=0
)

# Predicción en validación
results = model.predict(source='C:/Users/juanf/Desktop/Large-License-Plate-Detection-Dataset/images/val', save=True)


In [2]:
from ultralytics import YOLO
import cv2
from collections import defaultdict
import csv
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
import numpy as np

# ============================================================================
# CONFIGURACIÓN DEL MODELO smolVLM para OCR
# ============================================================================
print("Cargando modelo smolVLM...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "HuggingFaceTB/SmolVLM-Instruct"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
print(f"Modelo cargado en: {device}")

# ============================================================================
# FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON smolVLM
# ============================================================================
def extraer_texto_matricula(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando smolVLM
    """
    try:
        # Extraer ROI de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        if roi.size == 0:
            return ""
        
        # Convertir de BGR (OpenCV) a RGB (PIL)
        roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(roi_rgb)
        
        # Prompt específico para lectura de matrículas
        prompt = "Read the license plate number in this image. Only output the alphanumeric characters you see, without spaces or additional text."
        
        # Preparar entrada para el modelo
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        # Procesar con smolVLM
        text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=[text], images=[pil_image], return_tensors="pt")
        inputs = inputs.to(device)
        
        # Generar texto
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False
            )
        
        # Decodificar resultado
        generated_texts = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )
        
        # Extraer solo el texto de la respuesta
        texto = generated_texts[0].split("Assistant:")[-1].strip()
        
        # Limpiar el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in texto if c.isalnum()).upper()
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula: {e}")
        return ""

# ============================================================================
# FUNCIÓN PARA ASOCIAR MATRÍCULAS CON VEHÍCULOS
# ============================================================================
def asociar_matricula_con_vehiculo(plate_box, vehicle_boxes):
    """
    Encuentra el vehículo que contiene o está más cerca de la matrícula
    Retorna el índice del vehículo o None si no hay asociación
    """
    px1, py1, px2, py2 = plate_box
    plate_center_x = (px1 + px2) / 2
    plate_center_y = (py1 + py2) / 2
    
    mejor_vehiculo = None
    mejor_distancia = float('inf')
    
    for idx, vbox in enumerate(vehicle_boxes):
        vx1, vy1, vx2, vy2 = vbox
        
        # Verificar si la matrícula está dentro del vehículo
        if vx1 <= plate_center_x <= vx2 and vy1 <= plate_center_y <= vy2:
            return idx
        
        # Calcular distancia al centro del vehículo
        vcenter_x = (vx1 + vx2) / 2
        vcenter_y = (vy1 + vy2) / 2
        distancia = np.sqrt((plate_center_x - vcenter_x)**2 + (plate_center_y - vcenter_y)**2)
        
        if distancia < mejor_distancia:
            mejor_distancia = distancia
            mejor_vehiculo = idx
    
    # Solo asociar si la distancia es razonable (menos de 200 píxeles)
    if mejor_distancia < 200:
        return mejor_vehiculo
    return None

# ============================================================================
# CARGA DE MODELOS YOLO
# ============================================================================
general = YOLO("yolo11n.pt")
matriculas = YOLO("runs/detect/yolo_matriculas/weights/best.pt")

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
video_path = "C0142.MP4"
output_path = "detecciones_combinadas_final.mp4"
csv_path = "detecciones_tracking_matriculas.csv"
tracker = "bytetrack.yaml"

conf_general = 0.5
conf_plate = 0.3
classes_general = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck

# OPTIMIZACIÓN: Procesar OCR solo cada N frames
OCR_CADA_N_FRAMES = 5  # Ajusta este valor (5 = cada 5 frames, 10 = más rápido pero menos preciso)

# ============================================================================
# INICIALIZACIÓN
# ============================================================================
writer = None
ids_por_clase = defaultdict(set)

# OPTIMIZACIÓN: Cache de matrículas por tracking ID
matriculas_cache = {}  # track_id -> {'texto': str, 'confianza': float, 'bbox': tuple}

# Crear archivo CSV
csv_file = open(csv_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    'fotograma',
    'tipo_objeto',
    'confianza',
    'identificador_tracking',
    'x1', 'y1', 'x2', 'y2',
    'tiene_matricula',
    'confianza_matricula',
    'mx1', 'my1', 'mx2', 'my2',
    'texto_matricula'
])

# ============================================================================
# PROCESAMIENTO DEL VIDEO
# ============================================================================
print("Iniciando procesamiento del video...")
print(f"Optimizaciones activas:")
print(f"  - OCR cada {OCR_CADA_N_FRAMES} frames")
print(f"  - Cache de matrículas por tracking ID")
print(f"  - Sin visualización en tiempo real")

results_stream = general.track(
    source=video_path,
    tracker=tracker,
    classes=classes_general,
    conf=conf_general,
    persist=True,
    stream=True
)

for frame_num, r in enumerate(results_stream):
    if frame_num % 10 == 0:
        print(f"Procesando fotograma {frame_num}...")
    
    # Obtener frame original
    frame = r.orig_img.copy()
    
    # Inicializar escritor de video
    if writer is None:
        h, w = frame.shape[:2]
        fps = 30
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    
    # Dibujar tracking
    tracked_frame = r.plot()
    
    # Recopilar información de objetos trackeados
    objetos_trackeados = []
    if hasattr(r, "boxes") and r.boxes is not None:
        for box in r.boxes:
            if box.id is None:
                continue
            
            cls = int(box.cls)
            track_id = int(box.id)
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            ids_por_clase[cls].add(track_id)
            
            objetos_trackeados.append({
                'cls': cls,
                'track_id': track_id,
                'conf': conf,
                'bbox': (x1, y1, x2, y2),
                'tipo': general.names[cls]
            })
    
    # OPTIMIZACIÓN: Solo detectar matrículas y hacer OCR cada N frames
    matriculas_detectadas = []
    annotated = tracked_frame.copy()
    
    if frame_num % OCR_CADA_N_FRAMES == 0:
        # Detectar matrículas
        res_plate = matriculas(frame, conf=conf_plate, verbose=False)[0]
        
        if hasattr(res_plate, "boxes") and res_plate.boxes is not None:
            for pbox in res_plate.boxes:
                x1, y1, x2, y2 = map(int, pbox.xyxy[0].tolist())
                conf_plate_det = float(pbox.conf)
                
                # Extraer texto con smolVLM
                texto_matricula = extraer_texto_matricula(frame, x1, y1, x2, y2)
                
                matriculas_detectadas.append({
                    'bbox': (x1, y1, x2, y2),
                    'conf': conf_plate_det,
                    'texto': texto_matricula
                })
                
                # Dibujar solo el rectángulo de la matrícula (sin texto)
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)
    else:
        # En frames intermedios, solo dibujar las matrículas del cache
        for track_id, mat_info in matriculas_cache.items():
            # Buscar si este vehículo sigue presente en el frame actual
            for obj in objetos_trackeados:
                if obj['track_id'] == track_id and obj['tipo'] in ['car', 'motorcycle', 'bus', 'truck']:
                    # Dibujar rectángulo aproximado (basado en cache)
                    if 'bbox' in mat_info:
                        x1, y1, x2, y2 = mat_info['bbox']
                        cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    break
    
    # Asociar matrículas con vehículos y escribir en CSV
    matriculas_asociadas = set()
    
    for obj in objetos_trackeados:
        mejor_matricula = None
        
        # Si es un vehículo, buscar matrícula asociada
        if obj['tipo'] in ['car', 'motorcycle', 'bus', 'truck']:
            track_id = obj['track_id']
            
            # OPTIMIZACIÓN: Primero verificar si ya tenemos la matrícula en cache
            if track_id in matriculas_cache:
                mej

c:\Users\mario\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo smolVLM...


c:\Users\mario\anaconda3\envs\VC_P4\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mario\.cache\huggingface\hub\models--HuggingFaceTB--SmolVLM-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\mario\anaconda3\envs\VC_P4\lib\site-packages\transformers\models\auto\modeling_auto.py:2284: 

KeyboardInterrupt: 